# 11 — Karar Motoru

**Hedef:** IoT sensör verisi → sulama kararı + verim tahmini + ekonomik analiz

```
Gün 45 | Buğday | 10 dönüm
  Durum:          KRİTİK
  Gerekli su:     10.710 litre
  Sulamazsan:     3.900 kg/ha
  Sularsanız:     4.800 kg/ha
  Verim kazancı:  900 kg/ha × 6 TL = 5.400 TL
  Su maliyeti:    10.710 × 0.003 = 32 TL
  Net kazanç:     5.368 TL → SULA
```

**Girdi:** IoT sensör okumaları (gerçek veya simüle)

**Çıktı:** Aktüatör komutu + ekonomik özet

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

PROCESSED = Path('../data/processed')
MODELS    = Path('../outputs/models')
FIGS      = Path('../outputs/figures/decision_engine')
FIGS.mkdir(parents=True, exist_ok=True)

print('Hazır.')

## 1. Profil ve modelleri yükle

In [ ]:
# Günlük optimal nem profilleri
with open(PROCESSED / 'crop_daily_moisture_profile.json') as f:
    moisture_profiles = json.load(f)

# IoT verim modelleri
iot_models = {
    30: joblib.load(MODELS / 'iot' / 'iot_lgbm_day30.joblib'),
    60: joblib.load(MODELS / 'iot' / 'iot_lgbm_day60.joblib'),
    90: joblib.load(MODELS / 'iot' / 'iot_lgbm_day90.joblib'),
}

# Ürün fiyatları (TL/kg)
CROP_PRICES = {
    'wheat': 8.5, 'barley': 7.0, 'maize': 9.0, 'chickpea': 22.0,
    'rapeseed': 18.0, 'sunflower': 20.0, 'soybean': 19.0, 'cotton': 25.0,
    'potato': 6.0, 'sugarbeet': 3.5, 'rice': 14.0, 'tobacco': 45.0,
}
DEFAULT_PRICE = 10.0

WATER_COST_PER_M3 = 2.5  # TL/m³

# crop_te = target encoding (eğitimde kullanılan ortalama harvest_twso per bitki)
CROP_TE = {
    'barley': 4850.5, 'cassava': 1106.6, 'chickpea': 61.0, 'cotton': 545.1,
    'cowpea': 438.8, 'fababean': 4096.8, 'groundnut': 224.7, 'maize': 2179.9,
    'millet': 672.0, 'mungbean': 523.5, 'pigeonpea': 1924.9, 'potato': 2367.9,
    'rapeseed': 1539.9, 'rice': 565.0, 'seed_onion': 5638.9, 'sorghum': 1161.9,
    'soybean': 1069.3, 'sugarbeet': 4436.6, 'sunflower': 601.9,
    'sweetpotato': 2112.3, 'tobacco': 11.8, 'wheat': 4900.8,
}

print(f'{len(moisture_profiles)} bitki profili yüklendi.')
print(f'{len(iot_models)} IoT modeli yüklendi.')

## 2. Karar motoru fonksiyonu

In [ ]:
def get_model_for_day(season_day: int):
    if season_day <= 45:
        return iot_models[30], 30
    elif season_day <= 75:
        return iot_models[60], 60
    else:
        return iot_models[90], 90


def build_features(crop, season_day, mean_temp, total_precip, mean_humidity,
                   mean_soil_temp, soil_moisture, max_lai, lat, lon, elev, year, wav=50):
    return pd.DataFrame([{
        'mean_temp':          mean_temp,
        'total_precip':       total_precip,
        'mean_humidity':      mean_humidity,
        'mean_soil_temp':     mean_soil_temp,
        'mean_soil_moisture': soil_moisture,
        'max_lai':            max_lai,
        'season_days':        season_day,
        'latitude':           lat,
        'longitude':          lon,
        'elevation':          elev,
        'year':               year,
        'WAV':                wav,
        'crop_te':            CROP_TE.get(crop, 1000.0),  # target encoding, not integer
    }])


def irrigation_decision(
    crop: str,
    season_day: int,
    current_sm: float,
    mean_temp: float,
    total_precip: float,
    mean_humidity: float,
    mean_soil_temp: float,
    max_lai: float,
    lat: float,
    lon: float,
    elev: float,
    year: int,
    field_area_m2: float,
    root_depth: float = 0.30,
    wav: int = 50,
) -> dict:

    if crop not in moisture_profiles:
        return {'error': f'{crop} profili bulunamadı.'}

    p   = moisture_profiles[crop]
    idx = min(season_day, p['max_day'])

    target_sm   = p['optimal_sm'][idx]
    low_sm      = p['low_sm'][idx]
    critical_sm = p['critical_sm'][idx]
    upper_sm    = p['upper_sm'][idx]

    # Alarm
    if current_sm < critical_sm:
        alarm = 'KRİTİK'
    elif current_sm < low_sm:
        alarm = 'SULA'
    elif current_sm > upper_sm:
        alarm = 'DURDUR'
    else:
        alarm = 'NORMAL'

    # Karar
    if alarm == 'KRİTİK':
        decision = 'SULA (zorunlu)'
    elif alarm == 'SULA':
        decision = 'SULA'
    elif alarm == 'DURDUR':
        decision = 'SULAMA DURDUR'
    else:
        decision = 'BEKLEME'

    # Sulama miktarı (kademeli: SULA → low_sm, KRİTİK → target_sm)
    irr_target = target_sm if alarm == 'KRİTİK' else low_sm
    delta_sm   = max(0.0, irr_target - current_sm)
    litre      = delta_sm * field_area_m2 * root_depth * 1000
    water_cost = (litre / 1000) * WATER_COST_PER_M3

    # IoT verim tahmini
    model, checkpoint = get_model_for_day(season_day)
    feat = build_features(crop, season_day, mean_temp, total_precip,
                          mean_humidity, mean_soil_temp, current_sm,
                          max_lai, lat, lon, elev, year, wav)
    yield_forecast = max(0.0, float(model.predict(feat)[0]))

    # Tarihi verim referansı
    yield_optimal  = p['yield_optimal_kg_ha']
    yield_drought  = p['yield_drought_kg_ha']
    yield_at_risk  = max(0.0, yield_optimal - yield_drought)
    field_ha       = field_area_m2 / 10_000
    price          = CROP_PRICES.get(crop, DEFAULT_PRICE)
    season_risk_tl = yield_at_risk * field_ha * price

    return {
        'crop': crop, 'season_day': season_day, 'checkpoint': checkpoint,
        'alarm': alarm, 'decision': decision,
        'current_sm': round(current_sm, 4), 'target_sm': round(target_sm, 4),
        'low_sm': round(low_sm, 4), 'critical_sm': round(critical_sm, 4),
        'upper_sm': round(upper_sm, 4), 'irr_target_sm': round(irr_target, 4),
        'litre': round(litre, 0), 'water_cost_tl': round(water_cost, 2),
        'yield_forecast_kg_ha': round(yield_forecast, 1),
        'yield_optimal_kg_ha': round(yield_optimal, 1),
        'yield_drought_kg_ha': round(yield_drought, 1),
        'yield_at_risk_kg_ha': round(yield_at_risk, 1),
        'season_risk_tl': round(season_risk_tl, 2),
        'field_ha': round(field_ha, 2), 'price_tl_kg': price,
    }

print('Karar motoru hazır.')

## 3. Örnek karar — tek gün, tek tarla

In [ ]:
result = irrigation_decision(
    crop='wheat', season_day=45, current_sm=0.22,
    mean_temp=12.5, total_precip=95.0, mean_humidity=65.0,
    mean_soil_temp=10.0, max_lai=1.8,
    lat=39.0, lon=35.0, elev=950.0, year=2024,
    field_area_m2=10_000, root_depth=0.30, wav=50,
)

print(f"{'='*47}")
print(f"  Gün {result['season_day']} | {result['crop'].upper()} | {result['field_ha']} ha")
print(f"{'='*47}")
print(f"  Alarm:               {result['alarm']}")
print(f"  Karar:               {result['decision']}")
print(f"  ───────────────────────────────────────────")
print(f"  Mevcut nem:          {result['current_sm']:.3f} m³/m³")
print(f"  Hedef nem:           {result['irr_target_sm']:.3f} m³/m³  (sulama hedefi)")
print(f"  Kritik eşik:         {result['critical_sm']:.3f} m³/m³")
print(f"  ───────────────────────────────────────────")
print(f"  Gerekli su:          {result['litre']:,.0f} litre")
print(f"  Su maliyeti:         {result['water_cost_tl']:,.2f} TL")
print(f"  ───────────────────────────────────────────")
print(f"  IoT verim tahmini:   {result['yield_forecast_kg_ha']:,.1f} kg/ha")
print(f"  Tarihi optimum:      {result['yield_optimal_kg_ha']:,.1f} kg/ha  (üst %25 sezon)")
print(f"  Tarihi kurak:        {result['yield_drought_kg_ha']:,.1f} kg/ha  (alt %25 sezon)")
print(f"  Sezonluk risk ref.:  {result['season_risk_tl']:,.2f} TL  (bilgi amaçlı)")
print(f"{'='*47}")

## 4. Sezon simülasyonu — her gün karar ver

In [ ]:
np.random.seed(42)

CROP        = 'wheat'
FIELD_HA    = 1.0
TOTAL_DAYS  = 120

sm            = 0.30
cum_temp      = 0.0
cum_precip    = 0.0
cum_humidity  = 0.0
cum_soil_temp = 0.0
max_lai       = 0.0

records = []

for day in range(1, TOTAL_DAYS + 1):
    daily_temp     = 10 + day * 0.05 + np.random.normal(0, 1.5)
    daily_precip   = np.random.exponential(2.0)
    daily_humidity = 60 + np.random.normal(0, 5)
    daily_soil_t   = 8 + day * 0.04
    daily_lai      = min(3.0, 0.02 * day + np.random.normal(0, 0.05))

    cum_temp      = (cum_temp * (day - 1) + daily_temp) / day
    cum_precip   += daily_precip
    cum_humidity  = (cum_humidity * (day - 1) + daily_humidity) / day
    cum_soil_temp = (cum_soil_temp * (day - 1) + daily_soil_t) / day
    max_lai       = max(max_lai, daily_lai)

    sm = max(0.10, min(0.45, sm - 0.005 + daily_precip * 0.002 + np.random.normal(0, 0.004)))

    r = irrigation_decision(
        crop=CROP, season_day=day,
        current_sm=sm,
        mean_temp=cum_temp, total_precip=cum_precip,
        mean_humidity=cum_humidity, mean_soil_temp=cum_soil_temp,
        max_lai=max_lai,
        lat=39.0, lon=35.0, elev=950.0, year=2024,
        field_area_m2=FIELD_HA * 10_000,
        root_depth=0.30, wav=50,
    )
    r['sm_simulated'] = sm
    records.append(r)

    if r['decision'].startswith('SULA'):
        sm = r['irr_target_sm']   # sulandıktan sonra nemi güncelle

df_sim = pd.DataFrame(records)
irr_mask = df_sim['decision'].str.startswith('SULA')
print(f"Toplam sulama günü:   {irr_mask.sum()}")
print(f"Toplam su harcaması:  {df_sim['litre'].sum():,.0f} litre")
print(f"Toplam su maliyeti:   {df_sim['water_cost_tl'].sum():,.2f} TL")
print(f"Sezonluk risk (ref):  {df_sim['season_risk_tl'].iloc[0]:,.2f} TL")
df_sim[['season_day','alarm','decision','current_sm','irr_target_sm',
        'litre','water_cost_tl','yield_forecast_kg_ha']].head(10)

## 5. Görselleştirme — sezon boyunca nem + kararlar

In [ ]:
p    = moisture_profiles[CROP]
days = df_sim['season_day'].values

opt    = [p['optimal_sm'][d] for d in days]
low    = [p['low_sm'][d] for d in days]
crit   = [p['critical_sm'][d] for d in days]
upper  = [p['upper_sm'][d] for d in days]
high   = [p['high_sm'][d] for d in days]
curr_sm = df_sim['sm_simulated'].values

irr_days = df_sim[df_sim['decision'].str.startswith('SULA')]['season_day'].values

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# --- Üst: toprak nemi ---
ax1.fill_between(days, crit, upper, alpha=0.08, color='green', label='Güvenli aralık')
ax1.fill_between(days, low, high, alpha=0.15, color='steelblue', label='Optimum aralık')
ax1.plot(days, opt,     color='steelblue', lw=2,   label='Optimal hedef')
ax1.plot(days, crit,    color='red',    lw=1.2, ls='--', label='Kritik alt sınır')
ax1.plot(days, upper,   color='orange', lw=1.2, ls=':',  label='Üst sınır')
ax1.plot(days, curr_sm, color='black',  lw=1.5, label='Simüle nem (IoT)')
ax1.vlines(irr_days, ymin=0.08, ymax=0.50, color='cyan', alpha=0.5, lw=1.5, label='Sulama')
ax1.set_ylabel('Toprak nemi (m³/m³)')
ax1.set_title(f'{CROP.capitalize()} — Sezon Boyunca Toprak Nemi ve Sulama Kararları')
ax1.legend(fontsize=8, loc='upper right')
ax1.set_ylim(0.08, 0.50)

# --- Alt: IoT verim tahmini + tarihi referans bantları ---
forecast = df_sim['yield_forecast_kg_ha'].values
y_opt    = p['yield_optimal_kg_ha']
y_drought= p['yield_drought_kg_ha']

ax2.axhline(y_opt,    color='steelblue', lw=1.5, ls='--', label=f'Tarihi optimum ({y_opt:.0f} kg/ha)')
ax2.axhline(y_drought,color='red',       lw=1.5, ls='--', label=f'Tarihi kurak ({y_drought:.0f} kg/ha)')
ax2.fill_between(days, y_drought, y_opt, alpha=0.08, color='steelblue', label='Tarihi verim aralığı')
ax2.plot(days, forecast, color='black', lw=1.5, label='IoT verim tahmini')
ax2.set_xlabel('Gün (ekim sonrası)')
ax2.set_ylabel('Verim (kg/ha)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGS / 'season_decision_simulation.png', bbox_inches='tight')
plt.show()

## 6. Farklı bitkiler karşılaştırması — gün 60

In [ ]:
test_crops = ['wheat', 'maize', 'chickpea', 'sunflower', 'barley', 'cotton']
test_sm    = 0.22
test_day   = 60

rows = []
for crop in test_crops:
    r = irrigation_decision(
        crop=crop, season_day=test_day, current_sm=test_sm,
        mean_temp=14.0, total_precip=130.0, mean_humidity=58.0,
        mean_soil_temp=13.0, max_lai=2.2,
        lat=39.0, lon=35.0, elev=950.0,
        year=2024, field_area_m2=10_000, root_depth=0.30, wav=50,
    )
    rows.append(r)

df_crops = pd.DataFrame(rows)[[
    'crop', 'alarm', 'decision',
    'litre', 'water_cost_tl',
    'yield_forecast_kg_ha', 'yield_optimal_kg_ha', 'yield_drought_kg_ha',
    'yield_at_risk_kg_ha', 'season_risk_tl',
]]
df_crops.set_index('crop', inplace=True)
df_crops

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#e74c3c' if a == 'KRİTİK' else '#f39c12' if a == 'SULA' else '#2ecc71'
          for a in df_crops['alarm']]

# Sulama miktarı
axes[0].barh(df_crops.index, df_crops['litre'], color=colors)
axes[0].set_xlabel('Sulama miktarı (litre/ha)')
axes[0].set_title(f'Gün {test_day} — Sulama İhtiyacı')
axes[0].axvline(0, color='black', lw=0.8)

# Sezonluk risk
risk_colors = ['#c0392b' if v > 0 else '#27ae60' for v in df_crops['season_risk_tl']]
axes[1].barh(df_crops.index, df_crops['season_risk_tl'], color=risk_colors)
axes[1].set_xlabel('Sezonluk kuraklık riski (TL/ha)')
axes[1].set_title(f'Gün {test_day} — Verim Kayıp Riski')
axes[1].axvline(0, color='black', lw=0.8)

red_p = mpatches.Patch(color='#e74c3c', label='KRİTİK')
yel_p = mpatches.Patch(color='#f39c12', label='SULA')
grn_p = mpatches.Patch(color='#2ecc71', label='NORMAL')
axes[0].legend(handles=[red_p, yel_p, grn_p], fontsize=8)

plt.suptitle(f'Toprak nemi = {test_sm} m³/m³ | {len(test_crops)} bitki karşılaştırması', fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / 'crop_comparison_day60.png', bbox_inches='tight')
plt.show()

## 7. Aktüatör komutu üretimi

Gerçek sistemde bu fonksiyon MQTT'ye publish eder.

In [ ]:
def generate_actuator_command(
    result: dict,
    flow_rate_lpm: float = 500.0,  # litre/dakika (damla sulama pompası)
) -> dict:
    """
    Karar motoru çıktısından aktüatör komutu üretir.
    Gerçek sistemde MQTT broker'a publish edilir.
    """
    if result['decision'].startswith('SULA'):
        duration_min = result['litre'] / flow_rate_lpm
        cmd = {
            'action':       'OPEN',
            'duration_min': round(duration_min, 1),
            'litre':        result['litre'],
            'reason':       result['alarm'],
        }
    elif result['decision'] == 'SULAMA DURDUR':
        cmd = {'action': 'CLOSE', 'duration_min': 0, 'reason': 'DURDUR'}
    else:
        cmd = {'action': 'IDLE',  'duration_min': 0, 'reason': result['alarm']}

    # MQTT payload
    cmd['crop']       = result['crop']
    cmd['season_day'] = result['season_day']
    cmd['timestamp']  = pd.Timestamp.now().isoformat()
    return cmd


# Örnek
cmd = generate_actuator_command(result, flow_rate_lpm=500)
print(json.dumps(cmd, indent=2, ensure_ascii=False))